<a href="https://colab.research.google.com/github/theatu824/MSSP607-GitHub/blob/main/WeeklyModules/Week04/Week4_Pandas_Earthquakes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm
import statsmodels.api as se
import seaborn as sns
import datetime as dt
from sklearn.linear_model import LinearRegression
from scipy import stats
from io import StringIO
import itertools
pd.set_eng_float_format(accuracy=3, use_eng_prefix=True)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
from google.colab import userdata
import os
github_token = userdata.get('Neumann')
owner = 'theatu824' # Replace with the GitHub repository owner
repository = 'MSSP607-GitHub' # Replace with the GitHub repository name
clone_url = f'https://{github_token}@github.com/{owner}/{repository}.git'
# Clone the repository
import subprocess
subprocess.run(['git', 'clone', clone_url])
# Navigate into the cloned repository directory (optional, but often useful)
os.chdir(repository)
print(f"Changed directory to: {os.getcwd()}")

Changed directory to: /content/MSSP607-GitHub/MSSP607-GitHub


In [4]:
import pandas as pd
df = pd.read_csv("/content/MSSP607-GitHub/data/parsed.csv")
df.head()

,alert,cdi,code,detail,dmin,felt,gap,ids,mag,magType,...,status,time,title,tsunami,type,types,tz,updated,url,parsed_place
0,NaN,NaN,37389218,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.01,NaN,85.00,",ci37389218,",1.35,ml,...,automatic,1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.00,1539475395144,https://earthquake.usgs.gov/earthquakes/eventp...,California
1,NaN,NaN,37389202,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.02,NaN,79.00,",ci37389202,",1.29,ml,...,automatic,1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.00,1539475253925,https://earthquake.usgs.gov/earthquakes/eventp...,California
2,NaN,4.40,37389194,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.02,28.00,21.00,",ci37389194,",3.42,ml,...,automatic,1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0,earthquake,",dyfi,focal-mechanism,geoserve,nearby-cities,o...",-480.00,1539536756176,https://earthquake.usgs.gov/earthquakes/eventp...,California
3,NaN,NaN,37389186,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.03,NaN,39.00,",ci37389186,",0.44,ml,...,automatic,1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.00,1539475196167,https://earthquake.usgs.gov/earthquakes/eventp...,California
4,NaN,NaN,73096941,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.08,NaN,192.00,",nc73096941,",2.16,md,...,automatic,1539474716050,"M 2.2 - 10km NW of Avenal, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,scit...",-480.00,1539477547926,https://earthquake.usgs.gov/earthquakes/eventp...,California


a. Find the 95th percentile of earthquake magnitude in Japan using magType of 'mb'

In [5]:
japan = df[df["parsed_place"] == "Japan"]
japan_mb = japan[japan["magType"] == "mb"]
japan_mb["mag"].quantile(0.95)

np.float64(4.9)

b. Find the percentage of earthquakes in Indonesia that were coupled with tsunamis.

In [6]:
indonesia = df[df["parsed_place"] == "Indonesia"]
indonesia_tsu = df[df["tsunami"] == 1]
proportion = len(indonesia_tsu) / len(indonesia)
print(f"{proportion:.2%}")

41.50%


c. Get summary statistics for earthquakes in Nevada

In [7]:
nevada = df[df["parsed_place"] == "Nevada"]
nevada.describe()

,cdi,dmin,felt,gap,mag,mmi,nst,rms,sig,time,tsunami,tz,updated
count,15.00,681.00,15.00,681.00,681.00,1.00,681.00,681.00,681.00,681.00,681.00,681.00,681.00
mean,2.44,0.17,2.40,153.67,0.50,2.84,12.62,0.15,10.97,1538313805281.28,0.00,-480.00,1538401660548.00
std,0.50,0.17,4.63,68.74,0.70,NaN,9.87,0.08,19.61,596563746.20,0.00,0.00,601095107.44
min,2.00,0.00,1.00,29.14,-0.50,2.84,3.00,0.00,0.00,1537247268525.00,0.00,-480.00,1537306663881.00
25%,2.00,0.05,1.00,97.38,-0.10,2.84,6.00,0.11,0.00,1537854384162.00,0.00,-480.00,1537927505348.00
50%,2.20,0.11,1.00,149.14,0.40,2.84,10.00,0.15,2.00,1538279862199.00,0.00,-480.00,1538427614580.00
75%,2.90,0.23,1.00,199.72,0.90,2.84,16.00,0.19,12.00,1538821434704.00,0.00,-480.00,1538877980517.00
max,3.30,1.41,19.00,355.91,2.90,2.84,61.00,0.86,129.00,1539461038285.00,0.00,-480.00,1539482718719.00


d. Add a column to the dataframe indicating whether or not the earthquake happened in a country or US state that is on the Ring of Fire. Use Bolivia, Chile, Ecuador, Peru, Costa Rica, Guatemala, Mexico (be sure not to select New Mexico), Japan, Philippines, Indonesia, New Zealand, Antarctica (look for the Antarctic), Canada, Fiji, Alaska, Washington, California, Russia, Taiwan, Tonga, and the Kermadec Islands).

In [8]:
places_text = "Bolivia, Chile, Ecuador, Peru, Costa Rica, Guatemala, Mexico, Japan, Philippines, Indonesia, New Zealand, Canada, Fiji, Alaska, Washington, California, Russia, Taiwan, Tonga, Kermadec Islands"
ring_locations = places_text.split(", ")
is_listed = df["parsed_place"].isin(ring_locations)
is_antarctic = df["parsed_place"].str.contains("Antarctic", na = False)
df["ring_of_fire"] = is_listed | is_antarctic
df.head()

,alert,cdi,code,detail,dmin,felt,gap,ids,mag,magType,...,time,title,tsunami,type,types,tz,updated,url,parsed_place,ring_of_fire
0,NaN,NaN,37389218,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.01,NaN,85.00,",ci37389218,",1.35,ml,...,1539475168010,"M 1.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.00,1539475395144,https://earthquake.usgs.gov/earthquakes/eventp...,California,True
1,NaN,NaN,37389202,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.02,NaN,79.00,",ci37389202,",1.29,ml,...,1539475129610,"M 1.3 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.00,1539475253925,https://earthquake.usgs.gov/earthquakes/eventp...,California,True
2,NaN,4.40,37389194,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.02,28.00,21.00,",ci37389194,",3.42,ml,...,1539475062610,"M 3.4 - 8km NE of Aguanga, CA",0,earthquake,",dyfi,focal-mechanism,geoserve,nearby-cities,o...",-480.00,1539536756176,https://earthquake.usgs.gov/earthquakes/eventp...,California,True
3,NaN,NaN,37389186,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.03,NaN,39.00,",ci37389186,",0.44,ml,...,1539474978070,"M 0.4 - 9km NE of Aguanga, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,",-480.00,1539475196167,https://earthquake.usgs.gov/earthquakes/eventp...,California,True
4,NaN,NaN,73096941,https://earthquake.usgs.gov/fdsnws/event/1/que...,0.08,NaN,192.00,",nc73096941,",2.16,md,...,1539474716050,"M 2.2 - 10km NW of Avenal, CA",0,earthquake,",geoserve,nearby-cities,origin,phase-data,scit...",-480.00,1539477547926,https://earthquake.usgs.gov/earthquakes/eventp...,California,True


e. Calculate the number of earthquakes in the Ring of Fire locations and the number outside of them,

In [9]:
num_rof = len(df[df["ring_of_fire"] == True])
num_nrof = len(df[df["ring_of_fire"] == False])
summary = pd.DataFrame({
    "Region": ["Inside Ring of Fire", "Outside Ring of Fire"],
    "Earthquake count": [num_rof, num_nrof]
})

summary

,Region,Earthquake count
0,Inside Ring of Fire,7188
1,Outside Ring of Fire,2144


f. Find the tsunami count along the Ring of Fire.

In [10]:
len(df[(df["ring_of_fire"] == True) & (df["tsunami"] == 1)])

45